In [2]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np
from tqdm import tqdm  # Untuk progress bar

In [3]:
MODEL_NAME = "indolem/indobert-base-uncased"
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5
RANDOM_SEED = 42

In [4]:
# Set seed untuk reproducibility
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [5]:
df = pd.read_csv('../data/scrapped_spotify.csv', encoding='utf-8')
df = df.dropna(subset=['content', 'score'])

In [6]:
df['sentiment'] = df['score'].apply(lambda x: 0 if x <= 2 else (1 if x == 3 else 2))

In [7]:
# Split data dengan stratified sampling
df_train, df_temp = train_test_split(df, test_size=0.3, stratify=df['sentiment'], random_state=RANDOM_SEED)
df_val, df_test = train_test_split(df_temp, test_size=0.5, stratify=df_temp['sentiment'], random_state=RANDOM_SEED)

In [8]:
print(f"Data train: {len(df_train)}, validasi: {len(df_val)}, test: {len(df_test)}")

Data train: 711, validasi: 153, test: 153


In [9]:
class SpotifyDataset(Dataset):
    def __init__(self, reviews, targets, tokenizer, max_length=MAX_LEN):
        self.reviews = reviews
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.reviews)
    
    def __getitem__(self, idx):
        review = str(self.reviews[idx])
        target = self.targets[idx]
        
        encoding = self.tokenizer(
            review,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'targets': torch.tensor(target, dtype=torch.long)
        }


In [10]:
# Fungsi evaluasi
def evaluate_model(model, data_loader, device):
    model.eval()
    predictions = []
    actual_labels = []
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            targets = batch['targets'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            
            predictions.extend(preds)
            actual_labels.extend(targets.cpu().numpy())
    
    accuracy = accuracy_score(actual_labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(actual_labels, predictions, average='weighted')
    
    return accuracy, precision, recall, f1

In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [12]:
train_dataset = SpotifyDataset(df_train['content'].values, df_train['sentiment'].values, tokenizer)
val_dataset = SpotifyDataset(df_val['content'].values, df_val['sentiment'].values, tokenizer)
test_dataset = SpotifyDataset(df_test['content'].values, df_test['sentiment'].values, tokenizer)

In [13]:
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [14]:
# Set up model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [15]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=3  # 3 kelas: Negatif, Netral, Positif
)
model = model.to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indolem/indobert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [16]:
# Optimizer dengan learning rate scheduler
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = len(train_dataloader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

In [17]:
# Training loop
best_f1 = 0
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for batch in progress_bar:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets = batch['targets'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=targets)
        loss = outputs.loss
        total_loss += loss.item()
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        progress_bar.set_postfix({'loss': total_loss / (progress_bar.n + 1)})
    
    # Evaluasi per epoch
    val_accuracy, val_precision, val_recall, val_f1 = evaluate_model(model, val_dataloader, device)
    
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Validation - Accuracy: {val_accuracy:.4f}, F1-score: {val_f1:.4f}")
    
    # Save model jika performa lebih baik
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "best_model.pt")
        print("Model disimpan!")

Epoch 1/3: 100%|██████████| 45/45 [08:43<00:00, 11.63s/it, loss=0.765]
c:\Users\user\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Epoch 1/3
Validation - Accuracy: 0.7582, F1-score: 0.7225
Model disimpan!


Epoch 2/3: 100%|██████████| 45/45 [08:41<00:00, 11.60s/it, loss=0.644]
c:\Users\user\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Epoch 2/3
Validation - Accuracy: 0.7778, F1-score: 0.7256
Model disimpan!


Epoch 3/3: 100%|██████████| 45/45 [08:40<00:00, 11.58s/it, loss=0.556]
c:\Users\user\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Epoch 3/3
Validation - Accuracy: 0.7778, F1-score: 0.7408
Model disimpan!


In [18]:
# Load best model untuk evaluasi final
model.load_state_dict(torch.load("best_model.pt"))
test_accuracy, test_precision, test_recall, test_f1 = evaluate_model(model, test_dataloader, device)

c:\Users\user\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [19]:
print("\nHasil Evaluasi Final:")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall: {test_recall:.4f}")
print(f"F1-score: {test_f1:.4f}")


Hasil Evaluasi Final:
Accuracy: 0.8170
Precision: 0.7590
Recall: 0.8170
F1-score: 0.7851


In [20]:
def predict_sentiment(text, model, tokenizer, device):
    model.eval()
    encoding = tokenizer(
        text,
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        pred = torch.argmax(outputs.logits, dim=1).cpu().numpy()[0]
    
    sentiment_map = {0: "Negatif", 1: "Netral", 2: "Positif"}
    return sentiment_map[pred]

In [21]:
if __name__ == "__main__":
    sample_reviews = df_test['content'].values[:3]
    for review in sample_reviews:
        sentiment = predict_sentiment(review, model, tokenizer, device)
        print(f"Review: {review[:100]}...")
        print(f"Sentimen yang diprediksi: {sentiment}\n")

Review: G ngerti knp gbsa log in , udh berkali² pdhl jaringan ok...
Sentimen yang diprediksi: Negatif

Review: Tolong Spotify ditambahin fitur hapus history lagu yang baru diputar...
Sentimen yang diprediksi: Positif

Review: Aplikasi ini bagus dan menarik saya bisa dengar lagu yang saya inginkan saya juga berikan bintang 5👍...
Sentimen yang diprediksi: Positif

